In [42]:
using Pkg
using SpeedyWeather, GLMakie, Statistics

In [43]:
using SpeedyWeatherInternals.Utils
using LowerTriangularArrays
using SpeedyTransforms

In [44]:
include("TrenberthCallbacks.jl")  # loads module into Main
using .TrenberthCallbacks

In [45]:
# show whether the module actually has that constant
# : TRENBERTH_LONGNAMES in names(TrenberthCallbacks, all=true)

In [46]:
# spectral_grid = SpectralGrid(trunc=31, nlayers=8)
# model = PrimitiveWetModel(spectral_grid)

In [47]:
# using SpeedyWeather.Radiation
# using SpeedyWeather
# subtypes(AbstractShortwave)

spectral_grid = SpectralGrid(trunc=31, nlayers=8)
# model = PrimitiveWetModel(spectral_grid; shortwave_radiation=OneBandShortwave(spectral_grid))
model = PrimitiveWetModel(spectral_grid; shortwave_radiation=OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=true)))


# # get surface shortwave radiation down
# ssrd = simulation.diagnostic_variables.physics.surface_shortwave_down
# heatmap(ssrd,title="Surface shortwave radiation down [W/m^2]")

PrimitiveWetModel <: PrimitiveWet
├ spectral_grid: SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions.CPU}...
├ architecture: CPU{KernelAbstractions.CPU}
├ dynamics: Bool
├ geometry: Geometry{SpectralGrid{CPU{KernelAbstractions.CPU}, Spectrum{CPU{KernelAbstractions....
├ planet: Earth{Float32}
├ atmosphere: EarthAtmosphere{Float32}
├ coriolis: Coriolis{Vector{Float32}}
├ geopotential: Geopotential{Vector{Float32}}
├ adiabatic_conversion: AdiabaticConversion{Vector{Float32}}
├ particle_advection: Nothing
├ initial_conditions: InitialConditions{ZonalWind{Float32}, PressureOnOrography, JablonowskiTem...
├ forcing: Nothing
├ drag: Nothing
├ random_process: Nothing
├ tracers: Dict{Symbol, Tracer}
├ orography: EarthOrography{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussianGrid{...
├ land_sea_mask: EarthLandSeaMask{Float32, Field{Float32, 1, Vector{Float32}, OctahedralGaussia...
├ ocean: SlabOcean{Float32}
├ sea_ice: ThermodynamicSeaIce{Float32}
├ land: La

In [48]:
# simulation = initialize!(model)

In [49]:
# using SpeedyWeatherInternals
# run!(simulation, period=Week(1))

In [50]:
# spectral_grid = SpectralGrid()

# deciding between the cloud schemes:
# use without stratocumulus clouds
# sw_no_sc = OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=false))

# use with stratocumulus clouds
# sw_with_sc = OneBandShortwave(spectral_grid, clouds = DiagnosticClouds(spectral_grid; use_stratocumulus=true))

## Output variables

In [51]:
# see the model output structure:
model.output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ v: meridional wind [m/s]
 ├ humid: specific humidity [kg/kg]
 ├ temp: temperature [degC]
 ├ u: zonal wind [m/s]
 ├ mslp: mean sea-level pressure [hPa]
 └ vor: relative vorticity [s^-1]

In [52]:
add!(model, SpeedyWeather.RadiationOutput()...) # add radiation diagnostics to model output
SpeedyWeather.RadiationOutput()


(SpeedyWeather.OutgoingLongwaveRadiationOutput <: SpeedyWeather.AbstractOutputVariable
├ name::String = olr
├ unit::String = W/m^2
├ long_name::String = Outgoing longwave radiation
├ dims_xyzt::NTuple{4, Bool} = (true, true, false, true)
├ missing_value::Float64 = NaN
├ compression_level::Int64 = 1
├ shuffle::Bool = false
├ keepbits::Int64 = 7, SpeedyWeather.OutgoingShortwaveRadiationOutput <: SpeedyWeather.AbstractOutputVariable
├ name::String = osr
├ unit::String = W/m^2
├ long_name::String = Outgoing shortwave radiation
├ dims_xyzt::NTuple{4, Bool} = (true, true, false, true)
├ missing_value::Float64 = NaN
├ compression_level::Int64 = 1
├ shuffle::Bool = false
├ keepbits::Int64 = 7, SpeedyWeather.SurfaceShortwaveUpOutput <: SpeedyWeather.AbstractOutputVariable
├ name::String = sru
├ unit::String = W/m^2
├ long_name::String = Surface shortwave radiation up
├ dims_xyzt::NTuple{4, Bool} = (true, true, false, true)
├ missing_value::Float64 = NaN
├ compression_level::Int64 = 1
├ shuffle:

In [53]:
add!(model, SpeedyWeather.SurfaceFluxesOutput()...) # add surface flux diagnostics to model output

NetCDFOutput{Field{Float32, 1, Vector{Float32}, FullGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}}}
├ status: inactive/uninitialized
├ write restart file: true (if active)
├ interpolator: AnvilInterpolator{Float32, RingGrids.GridGeometry{OctahedralGaussianGrid{CPU{KernelAbstractions.CPU}, Vector{UnitRange{Int64}}, Vector{Int64}}, Vector{Float32}, Vector{Int64}}, RingGrids.AnvilLocator{Float32, Vector{Float32}, Vector{Int64}}}
├ path: output.nc (overwrite=false)
├ frequency: 21600 seconds
└┐ variables:
 ├ sru: Surface shortwave radiation up [W/m^2]
 ├ temp: temperature [degC]
 ├ srd: Surface shortwave radiation down [W/m^2]
 ├ mslp: mean sea-level pressure [hPa]
 ├ vor: relative vorticity [s^-1]
 ├ osr: Outgoing shortwave radiation [W/m^2]
 ├ v: meridional wind [m/s]
 ├ u: zonal wind [m/s]
 ├ shuf: Surface humidity flux (positive up) [kg/s/m^2]
 ├ albedo: albedo [1]
 ├ lrd: Surface longwave radiation down [W/m^2]
 ├ shf: Surface sensible heat flux (p

#### adding callbacks: 

In [54]:
# --------------------------
# helper: compute Trenberth diagnostics from diagn + model
# --------------------------
# - calc_trenberth_from_diagn
# function calc_trenberth_from_diagn(diagn, model; SumFlag::Bool=false)
#     fields = Dict(
#         # :LHF   => diagn.physics.surface_latent_heat_flux,
#         :LHF   => diagn.physics.surface_humidity_flux,
#         :SHF   => diagn.physics.sensible_heat_flux,
#         :SSRU  => diagn.physics.surface_shortwave_up,
#         :SLRU  => diagn.physics.surface_longwave_up,
#         :SSRD  => diagn.physics.surface_shortwave_down,
#         :SLRD  => diagn.physics.surface_longwave_down,
#         :OSR   => diagn.physics.outgoing_shortwave,
#         :OLR   => diagn.physics.outgoing_longwave,
#         :albedo=> diagn.physics.albedo
#     )

#     function calc_global_mean(field)
#         a = transform(field)
#         a00 = real(a[1])
#         return a00 / model.spectral_transform.norm_sphere
#     end

#     function calc_global_sum(field)
#         mean_val = calc_global_mean(field)
#         area = 4π * model.planet.radius^2
#         return mean_val * area
#     end

#     calcfun = SumFlag ? calc_global_sum : calc_global_mean

#     results = Dict{Symbol, Float64}()
#     for (k, f) in fields
#         try
#         if k == :LHF
#             results[k] = calcfun(f .* 2.5e6)  # convert moisture flux to latent heat flux (W/m² or W)
#         else
#             results[k] = calcfun(f)
#         end
#         catch err
#             @warn "calc_trenberth_from_diagn: could not compute $k: $err"
#             results[k] = NaN
#         end
#     end

#     results[:SW_net_sfc]  = results[:SSRD] - results[:SSRU]
#     results[:LW_net_sfc]  = results[:SLRD] - results[:SLRU]
#     results[:surface_net] = results[:SW_net_sfc] + results[:LW_net_sfc] - results[:LHF] - results[:SHF]

#     return results
# end


In [55]:
# Default long names for Trenberth variables
# const TRENBERTH_LONGNAMES = Dict(
#     :LHF => "Surface latent heat flux (W/m²)",
#     :SHF => "Surface sensible heat flux (W/m²)",
#     :SSRU => "Surface shortwave up (W/m²)",
#     :SLRU => "Surface longwave up (W/m²)",
#     :SSRD => "Surface shortwave down (W/m²)",
#     :SLRD => "Surface longwave down (W/m²)",
#     :OSR => "Outgoing shortwave radiation (TOA) (W/m²)",
#     :OLR => "Outgoing longwave radiation (TOA) (W/m²)",
#     :albedo => "Surface albedo",
#     :SW_net_sfc => "Surface net shortwave (W/m²)",
#     :LW_net_sfc => "Surface net longwave (W/m²)",
#     :surface_net => "Surface net energy (W/m²)"
# )

In [56]:
using Dates

# Convert various time types to Float64. Default unit = :seconds.
function time_to_float(t; unit::Symbol = :seconds)
    if t isa DateTime
        secs = Dates.datetime2unix(t)                     # seconds since Unix epoch
        return unit == :seconds ? Float64(secs) :
               unit == :days    ? Float64(secs / 86400.0) :
               error("unsupported unit: $unit")
    elseif t <: Dates.Period   # Day, Hour, Minute, etc.
        # Dates.value returns the integer magnitude in the Period's base units
        # For Day it returns number of days, for Hour number of hours, etc.
        # Convert to days or seconds depending on unit
        if unit == :days
            return float(Dates.value(t))
        elseif unit == :seconds
            # approximate: convert days/hours etc. to seconds using common ratios
            # We'll convert via Day/Hr/Minute explicitly for safety:
            if t isa Day
                return float(Dates.value(t) * 86400)
            elseif t isa Hour
                return float(Dates.value(t) * 3600)
            elseif t isa Minute
                return float(Dates.value(t) * 60)
            else
                # fallback: convert to days then seconds
                return float(Dates.value(Day(round(Int, Dates.value(t)))) * 86400)
            end
        else
            error("unsupported unit: $unit")
        end
    elseif t isa Number
        return float(t)
    else
        error("unsupported time type: $(typeof(t))")
    end
end


time_to_float (generic function with 1 method)

#### setting the callback and its schedule:

In [57]:
Base.@kwdef mutable struct TrenberthCallback <: SpeedyWeather.AbstractCallback
    timestep_counter::Int = 0
    data::Dict{Symbol, Vector{Float64}} = Dict{Symbol, Vector{Float64}}()
    times::Vector{Float64} = Float64[]          # elapsed seconds
    datetimes::Vector{DateTime} = DateTime[]    # original DateTime stamps
    start_time::Float64 = 0.0
    SumFlag::Bool = false
    var_longnames::Dict{Symbol,String} = TRENBERTH_LONGNAMES
    schedule::Schedule = Schedule()  # default: runs every timestep
end

# Constructor function to create instances with smart allocation
# function TrenberthCallback(; vars = [:LHF,:SHF,:SSRU,:SLRU,:SSRD,:SLRD,:OSR,:OLR,:albedo,:SW_net_sfc,:LW_net_sfc,:surface_net],
#                              SumFlag::Bool=false,
#                              nsteps::Int=0,
#                              var_longnames::Dict{Symbol,String}=TRENBERTH_LONGNAMES,
#                              schedule::Schedule=Schedule())
#     d = Dict{Symbol, Vector{Float64}}()
#     for v in vars
#         d[v] = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
#     end
#     times = nsteps > 0 ? Vector{Float64}(undef, nsteps + 1) : Float64[]
#     datetimes = nsteps > 0 ? Vector{DateTime}(undef, nsteps + 1) : DateTime[]
#     return TrenberthCallback(0, d, times, datetimes, 0.0, SumFlag, var_longnames, schedule)
# end


TrenberthCallback

In [58]:
methods(TrenberthCallback)

# 3 methods for type constructor:
 [1] TrenberthCallback(; timestep_counter, data, times, datetimes, start_time, SumFlag, var_longnames, schedule)
     @ In[57]:1
 [2] TrenberthCallback(timestep_counter::Int64, data::Dict{Symbol, Vector{Float64}}, times::Vector{Float64}, datetimes::Vector{DateTime}, start_time::Float64, SumFlag::Bool, var_longnames::Dict{Symbol, String}, schedule::Schedule)
     @ In[57]:2
 [3] TrenberthCallback(timestep_counter, data, times, datetimes, start_time, SumFlag, var_longnames, schedule)
     @ In[57]:2

In [59]:
# Run every timestep (default)
# cb = TrenberthCallback(SumFlag=false, nsteps=0)

# Run every day
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Day(1)))

# Run every 6 hours
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Hour(6)))

# Run every 12 hours
# cb = TrenberthCallback(SumFlag=false, nsteps=0, schedule=Schedule(every=Hour(12)))
cb = TrenberthCallback(SumFlag=false, timestep_counter=0, schedule=Schedule(every=Hour(12)))


LoadError: UndefVarError: `TRENBERTH_LONGNAMES` not defined in `Main`
Hint: It looks like two or more modules export different bindings with this name, resulting in ambiguity. Try explicitly importing it from a particular module, or qualifying the name with the module it should come from.

In [ ]:
# Pretty-print the long names
function show_var_names(cb::TrenberthCallback)
    for (k, long) in cb.var_longnames
        println(string(k), " → ", long)
    end
    return nothing
end

# assemble a DataFrame if DataFrames.jl is installed
function to_dataframe(cb::TrenberthCallback)
    try
        @eval using DataFrames
    catch
        error("DataFrames.jl not available. Install it with `using Pkg; Pkg.add(\"DataFrames\")`")
    end
    df = DataFrame(time = cb.datetimes)
    for (k, vec) in cb.data
        colname = get(cb.var_longnames, k, string(k))  # column name uses long name if available
        # ensure column identifier is a Symbol
        df[Symbol(colname)] = vec
    end
    return df
end

to_dataframe (generic function with 1 method)

In [ ]:
function SpeedyWeather.initialize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    
    # when initializing a scheduled callback also initialize its schedule!
    initialize!(cb.schedule, progn.clock)

    # Store the simulation start time for reference (convert DateTime to Float64 Unix timestamp)
    cb.start_time = Dates.datetime2unix(progn.clock.time)
    
    # Try to get nsteps, but if it doesn't work, just start with empty vectors
    try
        nsteps = progn.clock.nsteps
        # if our data dict vectors are empty or wrong size, (re)allocate
        for (k, v) in cb.data
            if isempty(v) || length(v) != nsteps + 1
                cb.data[k] = Vector{Float64}(undef, nsteps + 1)
            end
        end
        if isempty(cb.times) || length(cb.times) != nsteps + 1
            cb.times = Vector{Float64}(undef, nsteps + 1)
        end
        if isempty(cb.datetimes) || length(cb.datetimes) != nsteps + 1
            cb.datetimes = Vector{DateTime}(undef, nsteps + 1)
        end
    catch
        # If we can't get nsteps, just use dynamic push mode
        @info "Could not determine nsteps, using dynamic push mode"
    end

    # set counter to 1 and store initial conditions
    cb.timestep_counter = 1
    t0 = Dates.datetime2unix(progn.clock.time)  # Convert DateTime to Unix timestamp
    dt0 = progn.clock.time  # Get the original DateTime object
    # compute initial values using diagn
    res0 = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    for (k, v) in res0
        if haskey(cb.data, k) # check if key exists
            if length(cb.data[k]) > 0
                cb.data[k][1] = v
            else
                push!(cb.data[k], v)
            end
        else
            cb.data[k] = [v]
        end
    end

    # Store time relative to simulation start (in seconds) and DateTime
    if length(cb.times) > 0
        cb.times[1] = t0 - cb.start_time
        cb.datetimes[1] = dt0
    else
        push!(cb.times, t0 - cb.start_time)
        push!(cb.datetimes, dt0)
    end
    return nothing
end

In [ ]:
# --- callback! called every step (after the step completes) ---
function SpeedyWeather.callback!(cb::TrenberthCallback,
                                 progn::PrognosticVariables,
                                 diagn::DiagnosticVariables,
                                 model::AbstractModel)
    
    # scheduled callbacks start with this line to execute only when scheduled!
    # else escape immediately
    isscheduled(cb.schedule, progn.clock) || return nothing

    # increment step index
    cb.timestep_counter += 1
    
    # compute current diagnostics
    res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    
    # push new values to the arrays
    for (k, v) in res
        if !haskey(cb.data, k)
            # new key appeared: create vector and push
            cb.data[k] = [v]
        else
            # existing key: push to the vector
            push!(cb.data[k], v)
        end
    end
    
    # record model time relative to simulation start (in seconds) and DateTime
    # Convert DateTime to Unix timestamp, then subtract start_time to get elapsed seconds
    current_time = Dates.datetime2unix(progn.clock.time)
    push!(cb.times, current_time - cb.start_time)
    push!(cb.datetimes, progn.clock.time)  # Store the DateTime object
    return nothing
end

In [ ]:
# --- finalize (optional) ---
using Statistics  # Import mean function

# use the finalize! to clculate the mean values over the entire simulation:
function SpeedyWeather.finalize!(cb::TrenberthCallback,
                                   progn::PrognosticVariables,
                                   diagn::DiagnosticVariables,
                                   model::AbstractModel)
    # compute final diagnostics
    # res = calc_trenberth_from_diagn(diagn, model; SumFlag=cb.SumFlag)
    # for (k, v) in res
    #     if haskey(cb.data, k)
    #         push!(cb.data[k], v)
    #     else
    #         cb.data[k] = [v]
    #     end
    # end

    # compute the mean over the entire simulation for each variable and print
    # Note: we keep the original data vectors and just print the means
    println("\n=== Simulation Means ===")
    for (k, vec) in cb.data
        mean_val = mean(vec)
        println("Mean $k over simulation: $mean_val")
    end
    return nothing
end

adding the calbak to the model


In [ ]:
# A: add into the callbacks dict directly (works if model.callbacks is a Dict-like)
add!(model.callbacks, :trenberth => cb)

In [ ]:
keys(model.callbacks)              # should include :trenberth to make sure the callback was added correctly.

KeySet for a Dict{Symbol, SpeedyWeather.AbstractCallback} with 1 entry. Keys:
  :trenberth

In [ ]:
model.callbacks[:trenberth] === cb # should be true to make sure the callback was added correctly. 

true

In [ ]:
# SpeedyWeather.readable_secs

In [ ]:
sim = initialize!(model)   # this will call SpeedyWeather.initialize! on cb
run!(sim, period=Day(10))  # or your usual run invocation

[ Info: Could not determine nsteps, using dynamic push mode
Weather is speedy: 100%|██████████████████| Time: 0:00:04 (476.02 years/day)



=== Simulation Means ===
Mean LW_net_sfc over simulation: -71.71525210425968
Mean OLR over simulation: 243.11329287574404
Mean SSRD over simulation: 260.25633312406995
Mean SLRD over simulation: 330.0925045921689
Mean SHF over simulation: 50.96499978928339
Mean SSRU over simulation: 26.973105021885463
Mean SLRU over simulation: 401.80775669642856
Mean LHF over simulation: 25.41683679084388
Mean albedo over simulation: 0.12185197146165938
Mean SW_net_sfc over simulation: 233.28322810218447
Mean OSR over simulation: 76.87015024820964
Mean surface_net over simulation: 85.18613941779752


In [ ]:
cb.data # should contain the Trenberth variables collected during the simulation

Dict{Symbol, Vector{Float64}} with 12 entries:
  :LW_net_sfc  => [0.0, -72.6536, -66.436, -72.9242, -68.0592, -74.3138, -69.60…
  :OLR         => [0.0, 269.602, 268.448, 266.487, 264.436, 262.818, 261.05, 25…
  :SSRD        => [0.0, 261.356, 270.25, 268.445, 270.543, 271.467, 272.187, 27…
  :SLRD        => [0.0, 353.142, 354.526, 354.178, 353.379, 352.648, 351.596, 3…
  :SHF         => [0.0, 64.7518, 45.0865, 56.6862, 43.7429, 56.7295, 43.7139, 5…
  :SSRU        => [0.0, 31.5447, 21.7392, 33.0937, 21.8255, 33.7162, 22.0114, 3…
  :SLRU        => [0.0, 425.796, 420.962, 427.102, 421.438, 426.962, 421.203, 4…
  :LHF         => [0.0, -11.6067, -8.80666, 4.12649, -2.49309, 12.2631, 5.60018…
  :albedo      => [0.0, 0.127152, 0.127229, 0.127333, 0.127438, 0.12754, 0.1276…
  :SW_net_sfc  => [0.0, 229.811, 248.511, 235.352, 248.718, 237.75, 250.176, 23…
  :OSR         => [0.0, 96.8233, 77.773, 90.6341, 77.4719, 87.9748, 75.9084, 85…
  :surface_net => [0.0, 104.012, 145.795, 101.615, 139.409, 94

In [ ]:
size(cb.data[:LHF]) # should show the number of time steps + 1 (for initial condition)

(21,)

In [ ]:
cb.data[:LHF][:] # should show the last 10 values of the latent heat flux variable

21-element Vector{Float64}:
   0.0
 -11.606696749189831
  -8.806660787507077
   4.126485863215356
  -2.4930897253703863
  12.263147642414811
   5.6001813991636
  21.65711075039676
  14.843475868572275
  31.38783895361904
  24.349047262177848
  38.61402820556018
  31.35790510908891
  44.394978933572354
  36.79789870300232
  48.67134013301045
  40.91106357044478
  52.317214774268706
  44.632394456880704
  55.96860119935875
  48.76730704504178

In [ ]:
size(cb.times) # should show the number of time steps + 1 (for initial condition)

(21,)

In [ ]:
cb.times

21-element Vector{Float64}:
      0.0
  43200.0
  86400.0
 129600.0
 172800.0
 216000.0
 259200.0
 302400.0
 345600.0
 388800.0
 432000.0
 475200.0
 518400.0
 561600.0
 604800.0
 648000.0
 691200.0
 734400.0
 777600.0
 820800.0
 864000.0

In [ ]:
size(cb.datetimes) # should show the number of time steps + 1 (for initial condition)

(21,)

In [ ]:
cb.times

21-element Vector{Float64}:
      0.0
  43200.0
  86400.0
 129600.0
 172800.0
 216000.0
 259200.0
 302400.0
 345600.0
 388800.0
 432000.0
 475200.0
 518400.0
 561600.0
 604800.0
 648000.0
 691200.0
 734400.0
 777600.0
 820800.0
 864000.0

In [ ]:
cb.datetimes # to see the DateTime objects corresponding to the times

21-element Vector{DateTime}:
 2000-01-01T00:00:00
 2000-01-01T12:00:00
 2000-01-02T00:00:00
 2000-01-02T12:00:00
 2000-01-03T00:00:00
 2000-01-03T12:00:00
 2000-01-04T00:00:00
 2000-01-04T12:00:00
 2000-01-05T00:00:00
 2000-01-05T12:00:00
 2000-01-06T00:00:00
 2000-01-06T12:00:00
 2000-01-07T00:00:00
 2000-01-07T12:00:00
 2000-01-08T00:00:00
 2000-01-08T12:00:00
 2000-01-09T00:00:00
 2000-01-09T12:00:00
 2000-01-10T00:00:00
 2000-01-10T12:00:00
 2000-01-11T00:00:00

In [ ]:
cb.var_longnames

Dict{Symbol, String} with 12 entries:
  :surface_net => "Surface net energy (W/m²)"
  :SLRU        => "Surface longwave up (W/m²)"
  :LHF         => "Surface latent heat flux (W/m²)"
  :LW_net_sfc  => "Surface net longwave (W/m²)"
  :OLR         => "Outgoing longwave radiation (TOA) (W/m²)"
  :albedo      => "Surface albedo"
  :SSRD        => "Surface shortwave down (W/m²)"
  :SW_net_sfc  => "Surface net shortwave (W/m²)"
  :OSR         => "Outgoing shortwave radiation (TOA) (W/m²)"
  :SLRD        => "Surface longwave down (W/m²)"
  :SHF         => "Surface sensible heat flux (W/m²)"
  :SSRU        => "Surface shortwave up (W/m²)"